# Day 6 - Measuring every request (without breaking streaming)

**Big idea:** log time, tokens, and cost for every `/chat` call into Postgres - but the logging must never slow down or break the stream itself.

## 0. What I built (the map)

| File | Job |
|---|---|
| `days/day-006/app.py` | Day 3's `/chat` + `LoggingMiddleware` (raw ASGI) |
| `days/day-006/db.py` | asyncpg connection pool + `insert_request(...)` |
| `days/day-006/cost.py` | `estimate_cost(...)`, reusing Day 5's prices (imported, not retyped) |
| `days/day-006/test_middleware.py` | tests with the LLM **and** the database both faked |
| Postgres table `requests` | one row per `/chat` call |

How one request flows:

```
browser ──> LoggingMiddleware ──> FastAPI /chat ──> Ollama / Gemini
               │   (stopwatch starts; every chunk passes straight through to the browser)
               │
               └─ after the answer is fully sent:
                    tokens (tiktoken) -> cost (cost.py) -> insert row (db.py) -> Postgres
```

## 1. Middleware = the guard at the door

Middleware wraps **every** request. It sees the request come in and every piece of the response go out, so it's the natural place to start a stopwatch and write a log row.

## 2. ASGI in one breath

A Python web server talks to FastAPI through three things:
- `scope` - facts about the request (path, query string, ...)
- `receive` - call it to get incoming messages
- `send` - call it to push outgoing messages

A response is **one** `http.response.start` message (status + headers), then **one or more** `http.response.body` messages. A streaming response just sends many body messages over time.

Below: a tiny streaming app (3 words, 0.3 s apart) and a pretend browser that writes down *when* each chunk arrives. We'll reuse both for every experiment in this notebook.

In [1]:
import asyncio
import time

async def streaming_app(scope, receive, send):
    await send({"type": "http.response.start", "status": 200, "headers": []})
    for word in [b"one ", b"two ", b"three"]:
        await asyncio.sleep(0.3)                              # model thinking
        await send({"type": "http.response.body", "body": word, "more_body": True})
    await send({"type": "http.response.body", "body": b"", "more_body": False})


def make_scope(path="/chat", query=b""):
    return {"type": "http", "asgi": {"version": "3.0"}, "http_version": "1.1", "method": "GET",
            "scheme": "http", "path": path, "raw_path": path.encode(), "query_string": query,
            "root_path": "", "headers": [], "server": ("test", 80), "client": ("me", 1)}


async def client_view(app, scope=None):
    """Pretend browser: records the second at which each body chunk arrives."""
    scope = scope or make_scope()
    start = time.perf_counter()
    arrivals = []
    request_sent = False

    async def receive():
        nonlocal request_sent
        if not request_sent:
            request_sent = True
            return {"type": "http.request", "body": b"", "more_body": False}
        await asyncio.Event().wait()                          # browser stays connected

    async def send(message):
        if message["type"] == "http.response.body" and message.get("body"):
            arrivals.append((round(time.perf_counter() - start, 2), message["body"].decode()))

    await app(scope, receive, send)
    return arrivals

print("no middleware:", await client_view(streaming_app))

no middleware: [(0.3, 'one '), (0.6, 'two '), (0.9, 'three')]


Chunks arrive at ~0.3 s, ~0.6 s, ~0.9 s. That's what streaming *should* look like.

## 3. Why not `BaseHTTPMiddleware`? (measured, not assumed)

The Day 6 brief said `BaseHTTPMiddleware` buffers the whole response. On Day 7 I tested that claim against the Starlette actually installed here - and it was wrong in an interesting way. Let's run the same experiment: a streaming endpoint with the "obvious" timer wrapped around `call_next`.

In [2]:
import starlette
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import StreamingResponse
from starlette.routing import Route

async def slow_stream(request):
    async def words():
        for w in ["one ", "two ", "three"]:
            await asyncio.sleep(0.3)
            yield w
    return StreamingResponse(words(), media_type="text/event-stream")

timings = {}

class TimerMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        start = time.perf_counter()
        response = await call_next(request)
        timings["timer around call_next"] = round(time.perf_counter() - start, 2)
        return response

starlette_app = Starlette(routes=[Route("/chat", slow_stream)], middleware=[Middleware(TimerMiddleware)])

print("starlette", starlette.__version__)
print("browser saw chunks at:", await client_view(starlette_app))
print("the timer says:       ", timings)

starlette 1.6.0


browser saw chunks at: [(0.35, 'one '), (0.65, 'two '), (0.95, 'three')]
the timer says:        {'timer around call_next': 0.0}


Two findings:

1. **It does not buffer here** - chunks still arrive at ~0.3 / 0.6 / 0.9 s. So "BaseHTTPMiddleware destroys streaming" was the wrong reason.
2. **The timer lies** - it reads ~0.00 s for a response that took ~0.9 s. `call_next` returns as soon as the response *starts* (headers ready). The body streams *after* `dispatch` has already stopped the clock. It measures neither time-to-first-token nor total latency.

So to time a stream correctly you must hook the moment each chunk is actually **sent**. And if you try to fix it inside `BaseHTTPMiddleware` by reading the whole body first (e.g. to count tokens) and then returning it, *you* create the buffering. This is what that does to the user:

In [3]:
class ReadWholeBodyFirst:
    """Collects every message, THEN passes them on - what reading the full body before returning does."""
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        held = []
        async def hold(message):
            held.append(message)
        await self.app(scope, receive, hold)
        for message in held:
            await send(message)

print("read whole body first:", await client_view(ReadWholeBodyFirst(streaming_app)))

read whole body first: [(0.9, 'one '), (0.9, 'two '), (0.9, 'three')]


All three chunks land together at ~0.9 s: the user stares at a blank screen, then gets everything at once. Streaming destroyed.

## 4. The Day 6 middleware: wrap `send`, don't hold it

The fix: raw ASGI middleware that wraps `send` in a tiny function which **notes the time and passes each message straight through**. It keeps a private copy of the bytes for token-counting later - the browser already has the real ones.

In [4]:
class LoggingMiddleware:
    def __init__(self, app, log):
        self.app = app
        self.log = log

    async def __call__(self, scope, receive, send):
        start = time.perf_counter()
        ttfb = None
        chunks = []

        async def wrapped_send(message):
            nonlocal ttfb
            if message["type"] == "http.response.body":
                if ttfb is None:
                    ttfb = time.perf_counter() - start
                chunks.append(message.get("body", b""))
            await send(message)                        # forward immediately

        await self.app(scope, receive, wrapped_send)
        total = time.perf_counter() - start
        text = b"".join(chunks).decode()
        try:
            await self.log(ttfb_ms=round(ttfb * 1000), latency_ms=round(total * 1000), text=text)
        except Exception as exc:
            print("logging failed, request still fine:", exc)

async def print_log(**row):
    print("LOG ROW:", row)

print("browser saw:", await client_view(LoggingMiddleware(streaming_app, print_log)))

LOG ROW: {'ttfb_ms': 301, 'latency_ms': 903, 'text': 'one two three'}
browser saw: [(0.3, 'one '), (0.6, 'two '), (0.9, 'three')]


TTFB ~300 ms, total ~900 ms, and the browser still got every chunk on time.

The real version in `days/day-006/app.py` adds three things on top of this: it only watches `/chat`, it reads `prompt` and `provider` from the query string, and it skips logging for error responses (like `400 unknown provider`). Here it is, straight from the file:

In [5]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

source = (ROOT / "days" / "day-006" / "app.py").read_text()
start, end = source.index("class LoggingMiddleware"), source.index("app.add_middleware")
print(source[start:end])

class LoggingMiddleware:
    """Raw ASGI middleware: times TTFB/total latency and logs one Postgres
    row per /chat call, without ever buffering the streamed response.

    Deliberately not BaseHTTPMiddleware - see answers.md.
    """

    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http" or scope["path"] != "/chat":
            return await self.app(scope, receive, send)

        params = parse_qs(scope.get("query_string", b"").decode())
        prompt = params.get("prompt", [""])[0]
        provider = params.get("provider", ["ollama"])[0]

        start = time.perf_counter()
        ttfb = None
        status_code = None
        body_chunks: list[bytes] = []

        async def wrapped_send(message):
            nonlocal ttfb, status_code
            if message["type"] == "http.response.start":
                status_code = message["status"]
            elif message["type"] == "http.response.body":

## 5. Getting the text back out of the SSE bytes (and a bug I found)

The middleware sees raw SSE frames, not clean tokens. To count completion tokens it pulls out just the `data:` lines with a regex.

**Bug I found on Day 7:** sse-starlette ends every line with `\r\n`, but the original regex only stopped at `\n`, so every captured token kept a stray `\r`. Watch what that did to the count:

In [6]:
import re
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
frames = b"".join(
    f"event: message\r\ndata: {w}\r\n\r\n".encode()
    for w in ["Hello", " there", " friend", ",", " how", " are", " you", "?"]
)

buggy = re.compile(rb"^data:[ ]?(.*)$", re.MULTILINE)          # original Day 6 regex
fixed = re.compile(rb"^data:[ ]?(.*?)\r?$", re.MULTILINE)      # fixed version now in app.py

for name, pattern in [("buggy", buggy), ("fixed", fixed)]:
    text = b"".join(pattern.findall(frames)).decode()
    print(f"{name}: {text!r:<42} -> {len(enc.encode(text))} tokens")

buggy: 'Hello\r there\r friend\r,\r how\r are\r you\r?\r' -> 16 tokens
fixed: 'Hello there friend, how are you?'         -> 8 tokens


The stray `\r`s **doubled** the completion-token count (and so doubled the logged cost). Fixed in `app.py`, and the test now asserts the *exact* token count, so this can't sneak back in. Lesson: `> 0` is a weak assertion - it passed while the number was 2x wrong.

## 6. When Postgres is down

The log is written **after** the response has been fully sent, inside a `try/except`. A dead database costs a missing log row - never a broken answer. Observability must never take down the thing it observes.

In [7]:
async def broken_db(**row):
    raise ConnectionError("Postgres is down")

print("browser saw:", await client_view(LoggingMiddleware(streaming_app, broken_db)))

logging failed, request still fine: Postgres is down
browser saw: [(0.3, 'one '), (0.6, 'two '), (0.91, 'three')]


## 7. `db.py`: one table, one pool, one insert

The table:

```sql
CREATE TABLE requests (
    id                SERIAL PRIMARY KEY,
    ts                TIMESTAMPTZ NOT NULL DEFAULT now(),
    provider          TEXT NOT NULL,
    model             TEXT NOT NULL,
    prompt_tokens     INTEGER NOT NULL,
    completion_tokens INTEGER NOT NULL,
    ttfb_ms           DOUBLE PRECISION NOT NULL,
    latency_ms        DOUBLE PRECISION NOT NULL,
    cost_usd          DOUBLE PRECISION NOT NULL
);
```

Three ideas in `db.py`:
- **Connection pool** - opening a database connection is slow, so keep a few open and reuse them. The pool is created lazily on the first insert.
- **Placeholders (`$1 ... $7`)** - values are sent *separately* from the SQL text, so a prompt like `'); DROP TABLE requests; --` is stored as harmless text. Never build SQL with f-strings.
- **`DATABASE_URL` from `.env`** - connection details (with the password) stay out of git.

## 8. Cost: reuse, don't retype

`cost.py` imports Day 5's pricing dict instead of copying the numbers. A model Day 5 never priced (like local `llama3.2:3b`) returns `0.0`, meaning "untracked" - not a made-up price.

In [8]:
sys.path.insert(0, str(ROOT / "days" / "day-006"))
from cost import estimate_cost

for model in ["gpt-4o", "claude-sonnet-5", "llama3.2:3b"]:
    print(f"{model:<16} 100 in + 200 out tokens -> ${estimate_cost(100, 200, model):.6f}")

gpt-4o           100 in + 200 out tokens -> $0.002250
claude-sonnet-5  100 in + 200 out tokens -> $0.002200
llama3.2:3b      100 in + 200 out tokens -> $0.000000


## 9. Testing with no network and no database

A good test shouldn't need Ollama running or Postgres up. `test_middleware.py` swaps the real things for fakes:

- **Fake LLM:** replace the `"ollama"` entry in `TOKEN_SOURCES` with a tiny generator that yields `"pong", " ", "pong"`.
- **Fake database:** replace `insert_request` with an `AsyncMock`, then check *what it was called with* (provider, model, exact token counts, TTFB <= latency, cost).

One trap I hit: Day 3 and Day 6 both have an `app.py`, and both tests did `import app`. Python caches modules by name, so when running all tests together, Day 6's test silently got **Day 3's** app. The fix loads Day 6's file under a private name with `importlib.util.spec_from_file_location`.

Run the real tests:

In [9]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "days/day-006/test_middleware.py", "-q", "--color=no", "-p", "no:cacheprovider"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout.strip().splitlines()[-1])

2 passed in 0.62s


## 10. p95 latency: the "bad but normal" request

The **median** tells you what a typical request feels like. **p95** says: 95% of requests were faster than this. It catches the slow tail that users actually complain about.

In [10]:
import statistics

latencies_ms = [220, 240, 250, 260, 270, 280, 300, 310, 330, 350,
                360, 380, 400, 420, 450, 500, 600, 800, 1500, 4500]

p95 = statistics.quantiles(latencies_ms, n=100)[94]
print(f"median {statistics.median(latencies_ms):.0f} ms   p95 {p95:.0f} ms")

median 355 ms   p95 4350 ms


The median looks great; p95 shows some users waited seconds. Same idea in SQL against the real table:

```sql
SELECT provider,
       percentile_cont(0.95) WITHIN GROUP (ORDER BY latency_ms) AS p95_latency_ms
FROM requests
GROUP BY provider;
```

## 11. The prompt-token trap (coming in Week 2)

The middleware counts tokens on the **raw user prompt** from the URL. Once a system prompt and few-shot examples get added *inside* the app, the real prompt is much bigger - but the logged number doesn't change. The table quietly under-reports cost.

In [11]:
user_prompt = "Summarize: Machine M-14 temperature 92C exceeds threshold 85C"
system_prompt = "You are an industrial alert triage assistant. Reply in one short sentence."
few_shot = "\n".join(
    f"Alert: Machine M-{i:02d} energy draw 5.{i}kW exceeds 5.0kW\nSummary: M-{i:02d} is drawing too much power."
    for i in range(1, 6)
)

logged = len(enc.encode(user_prompt))
actual = len(enc.encode(system_prompt + "\n" + few_shot + "\n" + user_prompt))
print(f"middleware logs {logged} prompt tokens, model actually receives {actual} ({actual / logged:.1f}x more)")

middleware logs 17 prompt tokens, model actually receives 197 (11.6x more)


Fix for later: count tokens on the **exact string sent to the model**, not the raw user input.

## 12. Run it live yourself

```bash
brew services start postgresql@18        # local Postgres (no Docker on this machine)
ollama serve                             # local model, in another terminal
uv run uvicorn app:app --app-dir days/day-006 --port 8013
curl "http://127.0.0.1:8013/chat?prompt=Say%20hello%20in%20three%20words"
psql -h localhost -U rathnavel -d aej -c "SELECT * FROM requests;"
```

### (optional, live) The newest rows in Postgres

Rows 1 and 2 were logged *before* the `\r` fix, so their `completion_tokens` are doubled.

In [12]:
import os

import asyncpg
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

try:
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"), timeout=3)
    rows = await conn.fetch(
        "SELECT id, provider, model, prompt_tokens, completion_tokens, "
        "round(ttfb_ms) AS ttfb_ms, round(latency_ms) AS latency_ms, cost_usd "
        "FROM requests ORDER BY id DESC LIMIT 5"
    )
    await conn.close()
    for r in rows:
        print(dict(r))
    if not rows:
        print("table is empty - run a few /chat requests first")
except Exception as e:
    print("Skipped - Postgres isn't reachable:", type(e).__name__)

{'id': 2, 'provider': 'ollama', 'model': 'llama3.2:3b', 'prompt_tokens': 5, 'completion_tokens': 6, 'ttfb_ms': 228.0, 'latency_ms': 348.0, 'cost_usd': 0.0}
{'id': 1, 'provider': 'ollama', 'model': 'llama3.2:3b', 'prompt_tokens': 7, 'completion_tokens': 2, 'ttfb_ms': 4511.0, 'latency_ms': 4538.0, 'cost_usd': 0.0}


## 13. Day 6 exercises - short answers

**Why does `BaseHTTPMiddleware` fight you on SSE but not on JSON?**
`call_next` hands control back as soon as the response *starts*. For JSON that's basically the end anyway (one chunk). For SSE the whole answer streams *after* that point, so timing or reading the body there is wrong - and reading the full body before returning it buffers the stream yourself.

**SQL for p95 latency by provider?**
`percentile_cont(0.95) WITHIN GROUP (ORDER BY latency_ms)` with `GROUP BY provider` (section 10).

**Postgres is down - does the stream still work?**
Yes. The insert runs only after the response is fully sent, inside `try/except`. The user gets the full answer; we lose one log row. That's the behavior you want.

**Counting only the user's text as prompt tokens?**
Accurate today (no system prompt yet), but it silently under-counts once a system prompt and few-shot examples are added - by 10x+ in the demo above. Count what's actually sent to the model.

## 14. Day 6 recall quiz - short answers

- **(Day 5) Why do gpt-4o and claude-sonnet-5 count the same string differently?** Different vocabularies, built by BPE on different training text. Same string, different ruler.
- **(Day 5) Input or output - which costs more per token, and why?** Output. It's generated one token at a time (a full pass each); the input is processed in one parallel pass.
- **(Day 3) What triggers the `CancelledError` in the SSE generator?** sse-starlette notices the disconnect and cancels the task running the generator.
- **(Day 3) Why doesn't `request.is_disconnected()` fire?** sse-starlette's own listener reads the one-time "disconnect" message first, so my check never sees it.

## Recap

- Time a stream where bytes are **sent** (wrap `send`), never around `call_next` - that timer reads ~0.
- Raw ASGI middleware forwards each chunk immediately. Reading the whole body before returning = buffering.
- Log **after** the response, inside `try/except`. Logging must never break a request.
- SSE lines end in `\r\n` - parse carefully, and assert exact numbers in tests, not `> 0`.
- Fake the LLM and the DB in tests. Watch out for same-named modules across folders.
- Pool connections, use `$1` placeholders, reuse pricing constants. p95 shows the slow tail.